# Notebook 10 – Combining Datasets

Real data almost never lives in one single table. Customer details, order history, regional metadata, and model scores are usually spread across separate files or database tables — and before any analysis or model training can happen, they need to be **combined into one coherent dataset**. Pandas gives us five main tools for this: `concat()`, `merge()`, `join()`, the now-deprecated `append()`, and `combine_first()`.

### The Dataset We'll Use Throughout

We continue with the same online retail customer story from Notebooks 7, 8, and 9 — a single dataset that serves both a business analytics team and an AI/ML team building a **customer churn prediction model**.

- **Real-world / Business angle:** New customer batches arrive every quarter, region-level business metadata lives in a separate reference table, and different systems sometimes hold overlapping-but-incomplete customer records — all of this needs to be stitched together before management can trust a report.
- **AI/ML angle:** Before training the churn model, engineered features (region metadata, model scores from a previous run, reconciled customer attributes) must be combined with the base customer table into one clean feature matrix.

Let's rebuild the (clean) dataset first, then work through each combining technique.

In [14]:
import pandas as pd
import numpy as np
data = {
    "customer_id":   [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    "customer_name": ["Aarav", "Priya", "Rahul", "Sneha", "Vikram",
                       "Ananya", "Karthik", "Divya", "Manoj", "Lakshmi"],
    "region":        ["South", "North", "South", "West", "East",
                       "South", "North", "West", "East", "South"],
    "membership":    ["Gold", "Silver", "Gold", "Bronze", "Silver",
                       "Gold", "Bronze", "Gold", "Silver", "Bronze"],
    "total_orders":  [42, 15, 30, 5, 22, 60, 8, 35, 18, 3],
    "total_spend":   [125000, 32000, 98000, 8000, 45000,
                       210000, 12000, 87000, 39000, 4500],
    "last_order_value": [3200, 1500, 4200, 900, 2100,
                          5000, 1100, 3900, 1800, 700],
    "churn_risk":    ["Low", "Medium", "Low", "High", "Medium",
                        "Low", "High", "Low", "Medium", "High"]
}
df = pd.DataFrame(data)
df

,customer_id,customer_name,region,membership,total_orders,total_spend,last_order_value,churn_risk
0,101,Aarav,South,Gold,42,125000,3200,Low
1,102,Priya,North,Silver,15,32000,1500,Medium
2,103,Rahul,South,Gold,30,98000,4200,Low
3,104,Sneha,West,Bronze,5,8000,900,High
4,105,Vikram,East,Silver,22,45000,2100,Medium
5,106,Ananya,South,Gold,60,210000,5000,Low
6,107,Karthik,North,Bronze,8,12000,1100,High
7,108,Divya,West,Gold,35,87000,3900,Low
8,109,Manoj,East,Silver,18,39000,1800,Medium
9,110,Lakshmi,South,Bronze,3,4500,700,High


## 1. `pd.concat()`

### Concept Explanation
`pd.concat()` **stacks** DataFrames or Series together along an axis. With `axis=0` (default) it stacks rows on top of each other, useful when combining datasets that share the same columns. With `axis=1` it stacks columns side by side, aligning by index. Unlike `merge()`, `concat()` does not match rows using key-column values — it simply glues objects together and aligns on the shared axis, filling gaps with `NaN` (`join='outer'`, the default) or dropping them (`join='inner'`).

### Business + AI/ML Example
A new batch of customers signed up in Q2 and arrived as a separate export file. The business wants one single, up-to-date customer table. For the churn model, the same combined table becomes the base population the model is trained and scored on — so every new signup must be folded in before the next training run.

In [15]:
q2_signups = pd.DataFrame({
    "customer_id":   [111, 112],
    "customer_name": ["Rohan", "Neha"],
    "region":        ["North", "South"],
    "membership":    ["Silver", "Bronze"],
    "total_orders":  [4, 2],
    "total_spend":   [9500, 2600],
    "last_order_value": [1200, 500],
    "churn_risk":    ["Medium", "High"]
})
df_all_customers = pd.concat([df, q2_signups], ignore_index=True)
df_all_customers

,customer_id,customer_name,region,membership,total_orders,total_spend,last_order_value,churn_risk
0,101,Aarav,South,Gold,42,125000,3200,Low
1,102,Priya,North,Silver,15,32000,1500,Medium
2,103,Rahul,South,Gold,30,98000,4200,Low
3,104,Sneha,West,Bronze,5,8000,900,High
4,105,Vikram,East,Silver,22,45000,2100,Medium
5,106,Ananya,South,Gold,60,210000,5000,Low
6,107,Karthik,North,Bronze,8,12000,1100,High
7,108,Divya,West,Gold,35,87000,3900,Low
8,109,Manoj,East,Silver,18,39000,1800,Medium
9,110,Lakshmi,South,Bronze,3,4500,700,High


**Output Explanation:** The two Q2 signups (Rohan, Neha) were stacked directly beneath the original 10 customers, and `ignore_index=True` gave the combined table a clean, continuous index (0–11) instead of repeating the original 0–1 from `q2_signups`. This is the standard pattern for folding in a new batch of records with identical columns.

In [16]:
avg_order_value = pd.DataFrame({
    "avg_order_value": (df["total_spend"] / df["total_orders"]).round(2)
})
df_with_metrics = pd.concat([df, avg_order_value], axis=1)
df_with_metrics[["customer_name", "total_spend", "total_orders", "avg_order_value"]]

,customer_name,total_spend,total_orders,avg_order_value
0,Aarav,125000,42,2976.19
1,Priya,32000,15,2133.33
2,Rahul,98000,30,3266.67
3,Sneha,8000,5,1600.00
4,Vikram,45000,22,2045.45
5,Ananya,210000,60,3500.00
6,Karthik,12000,8,1500.00
7,Divya,87000,35,2485.71
8,Manoj,39000,18,2166.67
9,Lakshmi,4500,3,1500.00


**Output Explanation:** With `axis=1`, `concat()` lined up `avg_order_value` against `df` using their shared index and added it as a new column — this is column-wise stacking rather than row-wise. `avg_order_value` is exactly the kind of derived ratio feature a churn model would use.

## 2. `pd.merge()`

### Concept Explanation
`pd.merge()` is pandas' SQL-style **join** — it combines two DataFrames based on matching **values in one or more key columns**, not index position or simple stacking. The `how` parameter controls which rows survive: `'inner'` (default, only matches), `'left'`, `'right'`, or `'outer'`. This is the tool of choice whenever two tables are related through a shared key, like `customers` and a `region` reference table.

### Business + AI/ML Example
Regional business metadata (headquarters city, regional tax rate) lives in a separate reference table maintained by Finance. The business wants this attached to every customer row for reporting. For the churn model, `regional_tax_rate` is a candidate engineered feature — regions with different tax burdens may show different spending/churn patterns.

In [17]:
region_metadata = pd.DataFrame({
    "region": ["South", "North", "East", "West"],
    "hq_city": ["Chennai", "Delhi", "Kolkata", "Mumbai"],
    "regional_tax_rate": [0.18, 0.18, 0.12, 0.15]
})
df_with_region_info = pd.merge(df, region_metadata, on="region", how="left")
df_with_region_info[["customer_name", "region", "hq_city", "regional_tax_rate"]]

,customer_name,region,hq_city,regional_tax_rate
0,Aarav,South,Chennai,0.18
1,Priya,North,Delhi,0.18
2,Rahul,South,Chennai,0.18
3,Sneha,West,Mumbai,0.15
4,Vikram,East,Kolkata,0.12
5,Ananya,South,Chennai,0.18
6,Karthik,North,Delhi,0.18
7,Divya,West,Mumbai,0.15
8,Manoj,East,Kolkata,0.12
9,Lakshmi,South,Chennai,0.18


**Output Explanation:** Every customer row was matched to its region's metadata using `region` as the join key. A **left** join was used so that every customer from `df` is kept even if (hypothetically) a region were missing from `region_metadata` — the safest default when enriching a primary table with reference data.

In [18]:
partial_region_metadata = region_metadata[region_metadata["region"] != "West"]
left_merge = pd.merge(df, partial_region_metadata, on="region", how="left")
inner_merge = pd.merge(df, partial_region_metadata, on="region", how="inner")
print("Left join — West customers kept, hq_city/tax NaN:")
display(left_merge[left_merge["region"] == "West"][["customer_name", "region", "hq_city", "regional_tax_rate"]])
print("\nInner join — West customers dropped entirely:")
display(inner_merge[inner_merge["region"] == "West"])

Left join — West customers kept, hq_city/tax NaN:


,customer_name,region,hq_city,regional_tax_rate
3,Sneha,West,NaN,NaN
7,Divya,West,NaN,NaN



Inner join — West customers dropped entirely:


,customer_id,customer_name,region,membership,total_orders,total_spend,last_order_value,churn_risk,hq_city,regional_tax_rate


**Output Explanation:** With `partial_region_metadata` missing the West region, a **left** join still keeps West customers (Sneha, Divya) but with `NaN` metadata, while an **inner** join drops them completely because there's no matching West row on the right side. This shows exactly why `how` matters: it decides whether missing reference data hides a row or just leaves gaps in it.

## 3. `DataFrame.join()`

### Concept Explanation
`.join()` combines DataFrames primarily by their **index** (it can also join a calling DataFrame's column against another DataFrame's index via `on=`). It's a more concise, index-oriented cousin of `merge()` — internally, `.join()` actually calls `.merge()`. Default `how='left'`. It shines when your key naturally *is* the index, such as a lookup table indexed by ID.

### Business + AI/ML Example
The data science team ships churn-probability predictions from the **latest** model run as a small table indexed by `customer_id`. The business wants these scores attached to the customer table for a dashboard, and the same joined table is what a monitoring script would use to flag high-risk customers automatically.

In [6]:
latest_scores = pd.DataFrame({
    "churn_probability": [0.04, 0.35, 0.07, 0.75, 0.31, 0.02, 0.72, 0.09, 0.28, 0.83]
}, index=pd.Index([101, 102, 103, 104, 105, 106, 107, 108, 109, 110], name="customer_id"))

df_indexed = df.set_index("customer_id")
scored_customers = df_indexed.join(latest_scores)
scored_customers[["customer_name", "region", "churn_risk", "churn_probability"]]

,customer_name,region,churn_risk,churn_probability
customer_id,,,,
101,Aarav,South,Low,0.04
102,Priya,North,Medium,0.35
103,Rahul,South,Low,0.07
104,Sneha,West,High,0.75
105,Vikram,East,Medium,0.31
106,Ananya,South,Low,0.02
107,Karthik,North,High,0.72
108,Divya,West,Low,0.09
109,Manoj,East,Medium,0.28


**Output Explanation:** Because both `df_indexed` and `latest_scores` share the same index (`customer_id`), `.join()` matched them with a single, concise call — no explicit key column needed. Each customer now carries the latest model's `churn_probability` right next to their business `churn_risk` label, ready for a side-by-side comparison.

In [19]:
high_prob_but_labeled_low = scored_customers[
    (scored_customers["churn_probability"] > 0.5) & (scored_customers["churn_risk"] == "Low")
]
high_prob_but_labeled_low[["customer_name", "churn_risk", "churn_probability"]]

,customer_name,churn_risk,churn_probability
customer_id,,,


**Output Explanation:** This filter (empty in our sample data) is exactly the kind of automated check a monitoring dashboard runs after a `.join()`: any customer the model flags as high-risk (`churn_probability > 0.5`) but the business still labels 'Low' would show up here for manual review.

## 4. `.append()` — Deprecated, and the Modern Replacement

### Concept Explanation
`DataFrame.append()` used to add rows to a DataFrame, similar to a Python list's `.append()`. It was **deprecated in pandas 1.4** and **removed completely in pandas 2.0** — calling `df.append(...)` on a modern pandas install raises `AttributeError: 'DataFrame' object has no attribute 'append'`. It was removed because it encouraged appending inside loops (rebuilding the whole DataFrame every call — very slow at scale) and its behavior fully overlapped with `pd.concat()`, which is now the single, consistent replacement.

### Business + AI/ML Example
A new customer, Kavya, just signed up and needs to be added to the live customer table for today's dashboard refresh. In the churn-monitoring pipeline, new customer events arrive one at a time and must be appended to the growing training set — a place where the old `.append()`-in-a-loop pattern used to cause real slowdowns.

In [8]:

new_customer = pd.DataFrame([{
    "customer_id": 111, "customer_name": "Kavya", "region": "South",
    "membership": "Silver", "total_orders": 1, "total_spend": 1800,
    "last_order_value": 1800, "churn_risk": "Medium"
}])
df_updated = pd.concat([df, new_customer], ignore_index=True)
df_updated.tail(3)

,customer_id,customer_name,region,membership,total_orders,total_spend,last_order_value,churn_risk
8,109,Manoj,East,Silver,18,39000,1800,Medium
9,110,Lakshmi,South,Bronze,3,4500,700,High
10,111,Kavya,South,Silver,1,1800,1800,Medium


**Output Explanation:** The commented-out line shows legacy code that would crash on pandas 2.0+. The working replacement wraps the new customer as a one-row DataFrame and uses `pd.concat()`, producing the same result the old `.append()` used to — Kavya now appears as the 11th customer.

In [20]:
new_signups = [
    {"customer_id": 112, "customer_name": "Rohan", "region": "North", "membership": "Silver",
     "total_orders": 4, "total_spend": 9500, "last_order_value": 1200, "churn_risk": "Medium"},
    {"customer_id": 113, "customer_name": "Isha", "region": "East", "membership": "Bronze",
     "total_orders": 1, "total_spend": 950, "last_order_value": 950, "churn_risk": "High"},
]
df_efficient = pd.concat([df_updated, pd.DataFrame(new_signups)], ignore_index=True)
df_efficient.tail(3)

,customer_id,customer_name,region,membership,total_orders,total_spend,last_order_value,churn_risk
10,111,Kavya,South,Silver,1,1800,1800,Medium
11,112,Rohan,North,Silver,4,9500,1200,Medium
12,113,Isha,East,Bronze,1,950,950,High


**Output Explanation:** Instead of calling `concat()` once per new signup inside a loop (which would rebuild the whole table every iteration — the same performance trap the old `.append()` had), the new rows were collected into a plain Python list first and concatenated in a **single** call. Both signups (Rohan, Isha) landed correctly at the bottom in one efficient step.

## 5. `.combine_first()`

### Concept Explanation
`.combine_first()` merges two DataFrames/Series sharing the same index (and columns, for DataFrames), using the **calling object's values first** and **filling any `NaN` gaps with values from the other object**. It also unions in extra rows/columns that exist only in the other object. Think of it as *"use my data; where I have holes, fall back to yours."* It's ideal for reconciling two overlapping-but-incomplete versions of the same dataset — unlike `fillna()`, it handles differing indexes/columns gracefully.

### Business + AI/ML Example
The CRM system (primary source of customer records) has a few customers with a missing `churn_risk` label because a recent data-entry issue skipped that field. A secondary, slightly older export from the analytics warehouse has `churn_risk` filled in for those same customers. Before the churn model can be scored, every customer needs a non-missing `churn_risk`/label — this is a classic imputation-from-a-backup-source situation.

In [21]:
primary_source = df.set_index("customer_id").copy()
primary_source.loc[[104, 107, 110], "churn_risk"] = np.nan 
secondary_source = df.set_index("customer_id")[["churn_risk"]].copy()
secondary_source.loc[110, "churn_risk"] = np.nan
print("Primary source (has gaps):")
display(primary_source[["customer_name", "churn_risk"]])
print("\nSecondary source (mostly complete, one gap):")
display(secondary_source)

Primary source (has gaps):


,customer_name,churn_risk
customer_id,,
101,Aarav,Low
102,Priya,Medium
103,Rahul,Low
104,Sneha,NaN
105,Vikram,Medium
106,Ananya,Low
107,Karthik,NaN
108,Divya,Low
109,Manoj,Medium



Secondary source (mostly complete, one gap):


,churn_risk
customer_id,
101,Low
102,Medium
103,Low
104,High
105,Medium
106,Low
107,High
108,Low
109,Medium


**Output Explanation:** The primary CRM source is missing `churn_risk` for customers 104, 107, and 110. The secondary source has `churn_risk` for 104 and 107, but is itself missing it for 110 — neither source alone is complete.

In [11]:
reconciled = primary_source.combine_first(secondary_source)
reconciled[["customer_name", "churn_risk"]]

,customer_name,churn_risk
customer_id,,
101,Aarav,Low
102,Priya,Medium
103,Rahul,Low
104,Sneha,High
105,Vikram,Medium
106,Ananya,Low
107,Karthik,High
108,Divya,Low
109,Manoj,Medium


**Output Explanation:** `combine_first()` kept every non-missing value from `primary_source` untouched, and filled customers 104 and 107's missing `churn_risk` using the secondary source. Customer 110 is still `NaN`, since **neither** source had a value — `combine_first()` can only fill a gap if at least one of the two sources actually has data there, which correctly flags 110 as needing manual follow-up.